# insurance-governance

**Unified model governance for UK insurance pricing teams.**

This notebook shows the two things UK pricing teams need every time a model goes to committee:
an SS1/23-aligned (insurance adaptation) statistical validation report and a model risk tier assessment.
Both come from a single `pip install`.

The problem: validation tests (Gini, PSI, lift charts) and MRM governance packs (model cards,
risk tier scores) were always built separately and version-pinned separately.
This package merges them. One install, one import path.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burning-cost/insurance-governance/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q insurance-governance numpy

## Synthetic model outputs

We simulate a UK motor frequency model: 5,000 validation policies,
a Poisson-distributed observed claim count, and model-predicted frequencies.
The predictions are calibrated but imperfect — typical of a real model.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
n_val = 5_000

# Observed: Poisson claim counts on a motor book (mean ~8%)
y_true = rng.poisson(0.08, n_val).astype(float)

# Model predictions: calibrated but with some noise
y_pred = np.clip(rng.normal(0.08, 0.02, n_val), 0.002, None)

# Add some discrimination signal: sorted predictions correlate with outcome
# (realistic model has partial but not perfect lift)
sort_idx = np.argsort(y_pred)
y_true[sort_idx[4000:]] += rng.poisson(0.03, 1000)  # high-pred risks claim more

print(f"Validation set: {n_val:,} policies")
print(f"Observed mean freq: {y_true.mean():.4f}")
print(f"Predicted mean freq: {y_pred.mean():.4f}")
print(f"Calibration ratio: {y_true.mean() / y_pred.mean():.3f}")

## Step 1: Statistical validation (SS1/23-aligned)

`ModelValidationReport` runs the battery of tests the PRA expects:
discrimination (Gini), stability (PSI), calibration, and lift.
It produces a RAG-coded summary that goes directly into the Model Validation Pack.

The API is:
- `report.run()` — executes all tests, returns a list of `TestResult` objects
- `report.get_results()` — same, but cached after first call
- `report.get_rag_status()` — returns overall RAG (`'GREEN'`, `'AMBER'`, or `'RED'`)
- `report.generate(path)` — writes a self-contained HTML report
- `report.to_json(path)` — writes the JSON sidecar for MRM system ingestion
- `report.to_dict()` — serialises to a plain dict

In [ ]:
from insurance_governance import ModelValidationReport, ValidationModelCard

# ValidationModelCard accepts a simplified API: name/owner/features/target aliases
card = ValidationModelCard(
    name="Motor Frequency v3.2",
    version="3.2.0",
    purpose="Predict claim frequency for UK motor portfolio",
    methodology="CatBoost gradient boosting with Poisson objective",
    target="claim_count",
    features=["age", "vehicle_age", "area", "vehicle_group", "ncd_years"],
    limitations=["No telematics data", "Trained on 2020-2023 only"],
    owner="Pricing Team",
)

report = ModelValidationReport(model_card=card, y_val=y_true, y_pred_val=y_pred)

# Run all tests and get the overall RAG status
results = report.run()
rag = report.get_rag_status()

print(f"Overall RAG status: {rag}")
print(f"Number of tests run: {len(results)}")
print()
print("Test results:")
for r in results:
    status = 'PASS' if r.passed else 'FAIL'
    print(f"  {r.test_name:<35} [{status}]  {r.details[:60] if r.details else ''}")

## Step 2: Model Risk Tier scoring

`RiskTierScorer` applies an objective 0-100 composite score across six
dimensions and maps it to Tier 1 / 2 / 3. The tier drives sign-off requirements:
Tier 1 (Critical, ≥60 pts) needs a full Model Risk Committee pack,
Tier 3 (Informational, <30 pts) needs an owner attestation only.

The six dimensions are:
1. **Materiality** — GWP influenced (25 pts max)
2. **Complexity** — model architecture: `'low'`, `'medium'`, or `'high'` (20 pts max)
3. **Data quality** — external data use (10 pts max)
4. **Validation coverage** — months since last independent validation (10 pts max)
5. **Drift history** — monitoring trigger events in last 12 months (10 pts max)
6. **Regulatory exposure** — deployment status + regulatory use + customer-facing (25 pts max)

In [ ]:
from insurance_governance import MRMModelCard, RiskTierScorer, Assumption, Limitation

# MRMModelCard requires model_id, model_name, and version as mandatory fields
mrm_card = MRMModelCard(
    model_id="motor-freq-tppd-v32",
    model_name="Motor Frequency v3.2",
    version="3.2.0",
    model_class="pricing",
    model_type="GBM",
    intended_use="UK personal lines motor pricing — frequency component",
    target_variable="claim_count",
    distribution_family="Poisson",
    developer="Pricing Analytics",
    champion_challenger_status="champion",
    gwp_impacted=45_000_000,   # £45m book
    customer_facing=True,
    regulatory_use=False,
    monitoring_owner="Head of Pricing",
    rating_factors=["vehicle_age", "driver_age", "ncd_years", "area", "vehicle_group"],
    assumptions=[
        Assumption(
            description="Claim patterns in 2024+ match 2020-2023 training data",
            risk="MEDIUM",
            mitigation="Monthly PSI monitoring with threshold 0.10",
        ),
        Assumption(
            description="Policies are independent conditional on rating factors",
            risk="LOW",
        ),
    ],
    limitations=[
        Limitation(
            description="No telematics data integrated",
            impact="Higher variance in young driver predictions",
            population_at_risk="Drivers under 25",
        ),
        Limitation(
            description="Excludes commercial and fleet policies",
            impact="Not applicable to fleet underwriting",
        ),
    ],
)

scorer = RiskTierScorer()

# score() takes: gwp_impacted, model_complexity, deployment_status,
# regulatory_use, external_data, customer_facing
# plus optional: validation_months_ago, drift_triggers_last_year
tier_result = scorer.score(
    gwp_impacted=45_000_000,       # £45m book
    model_complexity="high",        # GBM with many features
    deployment_status="champion",   # in production
    regulatory_use=False,
    external_data=False,
    customer_facing=True,
    validation_months_ago=8,        # validated 8 months ago
    drift_triggers_last_year=1,     # one PSI breach in last year
)

print(f"Risk tier:    Tier {tier_result.tier} — {tier_result.tier_label}")
print(f"Score:        {tier_result.score:.1f} / 100")
print(f"Review:       {tier_result.review_frequency}")
print(f"Sign-off:     {tier_result.sign_off_requirement}")
print()
print("Dimension breakdown:")
for dim in tier_result.dimensions:
    contribution = (dim.score / dim.max_score) * tier_result.weights_used[dim.name]
    print(f"  {dim.name:<25} {contribution:5.1f}pts  —  {dim.rationale}")

## Step 3: Model inventory

`ModelInventory` maintains a persistent JSON registry of all models in production.
It tracks versions, tiers, owner, and last validation date.
This is what the Model Risk Committee looks at in a thematic review.

Key methods:
- `inventory.register(card, tier)` — add or update a model
- `inventory.list()` — return all models as a list of flat dicts (convert to DataFrame with `pl.DataFrame(inventory.list())`)
- `inventory.update_validation(model_id, ...)` — record validation outcome
- `inventory.due_for_review(within_days)` — models needing review soon
- `inventory.summary()` — high-level statistics

In [ ]:
import tempfile
import os
from insurance_governance import ModelInventory

# Use a temp file — in production this would be a shared drive or git-backed path
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
    inventory_path = f.name

inventory = ModelInventory(path=inventory_path)

# register(card, tier) — tier is a TierResult from RiskTierScorer.score()
inventory.register(mrm_card, tier_result)

# Record the most recent validation outcome
inventory.update_validation(
    model_id="motor-freq-tppd-v32",
    validation_date="2025-07-15",
    overall_rag="GREEN",
    next_review_date="2026-07-15",
    notes="Gini stable at 0.38, PSI 0.04, A/E 1.01",
)

# Add a simpler GLM model for comparison
mrm_card2 = MRMModelCard(
    model_id="home-freq-bldgs-v15",
    model_name="Home Buildings Frequency v1.5",
    version="1.5.0",
    model_class="pricing",
    model_type="GBM",
    intended_use="UK home buildings pricing — frequency component",
    target_variable="claim_count",
    distribution_family="Poisson",
    developer="Pricing Analytics",
    champion_challenger_status="champion",
    gwp_impacted=12_000_000,   # £12m book
    customer_facing=True,
    regulatory_use=False,
    monitoring_owner="Head of Pricing",
)

tier_result2 = scorer.score(
    gwp_impacted=12_000_000,
    model_complexity="low",      # simple GLM
    deployment_status="champion",
    regulatory_use=False,
    external_data=False,
    customer_facing=True,
    validation_months_ago=6,
    drift_triggers_last_year=0,
)

inventory.register(mrm_card2, tier_result2)
inventory.update_validation(
    model_id="home-freq-bldgs-v15",
    validation_date="2025-09-01",
    overall_rag="GREEN",
    next_review_date="2026-09-01",
)

# list() returns a list of flat dicts — easy to tabulate
rows = inventory.list()
print("Model inventory:")
print(f"{'model_id':<30} {'tier':<6} {'tier_label':<15} {'status':<12} {'rag':<8} {'next_review'}")
print("-" * 90)
for row in rows:
    print(
        f"{row['model_id']:<30} "
        f"{str(row['materiality_tier']):<6} "
        f"{row['tier_label']:<15} "
        f"{row['champion_challenger_status']:<12} "
        f"{row['overall_rag']:<8} "
        f"{row['next_review_date']}"
    )

print()
summary = inventory.summary()
print(f"Total models: {summary['total_models']}")
print(f"By tier: {summary['by_tier']}")
print(f"By RAG:  {summary['by_rag']}")
print(f"Overdue for review: {summary['overdue_count']}")

os.unlink(inventory_path)

## What you should see

- The validation report shows results for each test (Gini, calibration, PSI) with PASS/FAIL
  and an overall RAG status.
- Motor Frequency v3.2 (GBM, £45m book, in production) should score Tier 1 or Tier 2 —
  it is a large, complex, customer-facing model.
- Home Buildings v1.5 (GLM, £12m) should score Tier 2 or Tier 3.
- The inventory shows both models with tier, RAG, and next review date.

## API reference

### ModelValidationReport
| Method | Returns | Notes |
|--------|---------|-------|
| `run()` | `list[TestResult]` | Executes all tests |
| `get_results()` | `list[TestResult]` | Cached after first `run()` |
| `get_rag_status()` | `RAGStatus` | `'GREEN'`, `'AMBER'`, or `'RED'` |
| `generate(path)` | `Path` | Writes self-contained HTML |
| `to_json(path)` | `Path` | Writes JSON sidecar |
| `to_dict()` | `dict` | Serialise to plain dict |

### RiskTierScorer.score()
Required: `gwp_impacted`, `model_complexity` (`'low'`/`'medium'`/`'high'`),
`deployment_status`, `regulatory_use`, `external_data`, `customer_facing`.
Optional: `validation_months_ago`, `drift_triggers_last_year`.

### ModelInventory
| Method | Signature | Notes |
|--------|-----------|-------|
| `register(card, tier)` | `(ModelCard, TierResult)` | Add or update |
| `list(...)` | returns `list[dict]` | All models, filterable |
| `update_validation(model_id, ...)` | — | Record validation outcome |
| `due_for_review(within_days)` | returns `list[dict]` | Review schedule |
| `summary()` | returns `dict` | Counts by tier/status/RAG |

## Next steps

- **`GovernanceReport`** — generates a full Model Risk Committee pack as Markdown
- **`ReportGenerator`** — exports the validation report as self-contained HTML
- **`DiscriminationReport`** and **`StabilityReport`** — run individual test categories

**GitHub:** https://github.com/burning-cost/insurance-governance  
**PyPI:** https://pypi.org/project/insurance-governance/